# Secure Retail Data Lakehouse
### Celebal Technologies — Data Engineering Assignment

## Problem Statement
Retail operational systems (E-commerce, POS) continuously collect raw PII (names, addresses,
phone numbers, dates of birth) and PCI data (card numbers, CVV). Storing this in plain text
creates security vulnerabilities and violates compliance frameworks (**PCI-DSS**, **GDPR**, **DPDP**).
Internal analysts also don't need raw identities — only aggregate trends — so unrestricted
access violates the **principle of least privilege**.

## Dataset
This project uses the real **Superstore Sales Dataset** (9,994 orders, 793 unique customers)
as the transactional backbone. Since real-world e-commerce/POS systems also capture PII (email,
phone, date of birth) and PCI (card number, CVV) at checkout — fields the public Superstore
dataset doesn't include — these are enriched in **deterministically** per customer/order
(same customer always gets the same email/phone/DOB across orders; each order gets its own
card transaction), so the data behaves like a genuine operational export.

## What We Are Building
A secure, automated **static batch pipeline** that acts as a compliance filter between raw
retail data and analytics teams, using a progressively secure **Bronze → Silver → Gold**
lakehouse architecture:

| Layer | Contains | Who can access |
|-------|----------|-----------------|
| **Raw (Landing Zone)** | Superstore orders enriched with PII/PCI | No one (transient, deleted after Bronze ingestion) |
| **Bronze** | Near-raw data, CVV hard-dropped | `compliance_admin` only |
| **Silver** | PII masked, PCI tokenized + encrypted, features engineered | `data_engineer`, `data_scientist` (partial) |
| **Gold** | Fully aggregated, zero identifiers | `business_analyst` (everyone) |

### Security Controls Implemented
1. **Hard-Drop** — CVV is physically deleted at ingestion, never written to any layer.
2. **Masking** — names, emails, phone numbers partially redacted.
3. **Tokenization** — card numbers replaced with irreversible SHA-256 surrogate tokens (salted).
4. **Encryption** — the surrogate token is further encrypted at rest (Fernet/AES-128); only a
   role holding the key can decrypt it.
5. **Feature Engineering** — date of birth converted to age buckets, sales amounts bucketed.
6. **RBAC (Access Control)** — roles get scoped views enforced in code (simulating
   Azure Data Lake ACLs / Databricks Unity Catalog GRANTs in production).
---


## Step 0: Setup

In [18]:
%pip install faker cryptography pyarrow

In [19]:
import pandas as pd
import numpy as np
import hashlib
import random
from datetime import datetime
from faker import Faker
from cryptography.fernet import Fernet

pd.set_option('display.max_columns', None)


## Step 1: Raw Landing Zone — Superstore Enriched with PII/PCI

We start from the **real Superstore dataset** (Order ID, Customer Name, Sales, Profit, etc.)
and enrich it with realistic PII/PCI fields, deterministically tied to each customer/order —
representing data exactly as it would arrive from an e-commerce checkout / POS system,
**before any security processing**.


In [20]:
import os
print(os.getcwd())
print(os.listdir())

d:\Admin\Downloads\Secure_Retail_Lakehouse_Project\secure_lakehouse
['01_generate_raw_data.py', '02_bronze_ingestion.py', '03_silver_transformation.py', '04_gold_aggregation.py', '05_rbac_access_control.py', 'bronze', 'gold', 'keys', 'raw', 'README.md', 'Sample - Superstore.csv', 'Secure_Retail_Lakehouse.ipynb', 'silver']


In [21]:
import os
print(os.listdir("raw"))

['raw_retail_transactions.csv']


In [22]:
fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

superstore = pd.read_csv("Sample - Superstore.csv", encoding="latin1")
print(f"Real Superstore data: {superstore.shape}")
superstore.head(2)


Real Superstore data: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820


In [23]:
def deterministic_faker_for_customer(customer_id):
    seed = int(hashlib.md5(customer_id.encode()).hexdigest(), 16) % (2**32)
    lf = Faker()
    lf.seed_instance(seed)
    return lf

card_networks = ["4", "5", "6"]

def build_raw_dataset(df):
    df = df.copy()

    # Enrich PII per unique customer (consistent across their orders)
    customer_profiles = {}
    for cust_id, cust_name in df[["Customer ID", "Customer Name"]].drop_duplicates().values:
        cf = deterministic_faker_for_customer(cust_id)
        customer_profiles[cust_id] = {
            "email": f"{cust_name.lower().replace(' ', '.')}@{cf.free_email_domain()}",
            "phone_number": cf.phone_number(),
            "date_of_birth": cf.date_of_birth(minimum_age=18, maximum_age=75).strftime("%Y-%m-%d"),
        }
    df["email"] = df["Customer ID"].map(lambda c: customer_profiles[c]["email"])
    df["phone_number"] = df["Customer ID"].map(lambda c: customer_profiles[c]["phone_number"])
    df["date_of_birth"] = df["Customer ID"].map(lambda c: customer_profiles[c]["date_of_birth"])

    # Enrich PCI per order (each transaction has its own card swipe)
    card_numbers, card_expiries, cvvs = [], [], []
    for order_id in df["Order ID"]:
        seed = int(hashlib.md5(order_id.encode()).hexdigest(), 16) % (2**32)
        rnd = random.Random(seed)
        prefix = rnd.choice(card_networks)
        card_numbers.append(prefix + "".join(str(rnd.randint(0, 9)) for _ in range(15)))
        card_expiries.append(f"{rnd.randint(1,12):02d}/{rnd.randint(26,30)}")
        cvvs.append(f"{rnd.randint(100,999)}")
    df["card_number"], df["card_expiry"], df["cvv"] = card_numbers, card_expiries, cvvs

    df = df.rename(columns={
        "Row ID": "row_id", "Order ID": "order_id", "Order Date": "order_date",
        "Ship Date": "ship_date", "Ship Mode": "ship_mode", "Customer ID": "customer_id",
        "Customer Name": "full_name", "Segment": "segment", "Country": "country",
        "City": "city", "State": "state", "Postal Code": "postal_code", "Region": "region",
        "Product ID": "product_id", "Category": "category", "Sub-Category": "sub_category",
        "Product Name": "product_name", "Sales": "sales", "Quantity": "quantity",
        "Discount": "discount", "Profit": "profit",
    })
    return df

raw_df = build_raw_dataset(superstore)
raw_df.to_csv("raw/raw_retail_transactions.csv", index=False)
print(f"Raw enriched dataset: {raw_df.shape}")
raw_df[["order_id", "full_name", "email", "phone_number", "date_of_birth",
        "card_number", "cvv", "sales"]].head(3)


Raw enriched dataset: (9994, 27)


,order_id,full_name,email,phone_number,date_of_birth,card_number,cvv,sales
0,CA-2016-152156,Claire Gute,claire.gute@gmail.com,8235062526,2000-03-02,5527812096789655,860,261.96
1,CA-2016-152156,Claire Gute,claire.gute@gmail.com,8235062526,2000-03-02,5527812096789655,860,731.94
2,CA-2016-138688,Darrin Van Huff,darrin.van.huff@gmail.com,4754940413,1988-08-02,5465283994305638,192,14.62


⚠️ **Notice:** `full_name`, `email`, `phone_number`, `date_of_birth`, `card_number`, and
`cvv` are all in **plain text** here. This is exactly the exposure the pipeline exists to
eliminate.


## Step 2: Bronze Layer — Ingestion with Immediate Hard-Drop

The **first security checkpoint**. As data lands in the lake, the CVV — which PCI-DSS
prohibits storing under any circumstance, even encrypted — is dropped immediately, before
anything touches disk.


In [24]:
def ingest_to_bronze(raw_path, bronze_path):
    df = pd.read_csv(raw_path)

    # SECURITY CONTROL 1: HARD-DROP — CVV must never be persisted (PCI-DSS)
    if "cvv" in df.columns:
        df = df.drop(columns=["cvv"])
        print("[HARD-DROP] 'cvv' column permanently dropped at ingestion.")

    df["_ingestion_timestamp"] = datetime.utcnow().isoformat()
    df["_source_system"] = "retail_pos_ecommerce"
    df["_layer"] = "bronze"

    df.to_parquet(bronze_path, index=False)
    print(f"[BRONZE] {df.shape[0]} rows, {df.shape[1]} columns -> {bronze_path}")
    return df

bronze_df = ingest_to_bronze("raw/raw_retail_transactions.csv", "bronze/retail_bronze.parquet")
bronze_df.head(3)


[HARD-DROP] 'cvv' column permanently dropped at ingestion.


C:\Users\Admin\AppData\Local\Temp\ipykernel_24084\664201519.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df["_ingestion_timestamp"] = datetime.utcnow().isoformat()


[BRONZE] 9994 rows, 29 columns -> bronze/retail_bronze.parquet


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,full_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,email,phone_number,date_of_birth,card_number,card_expiry,_ingestion_timestamp,_source_system,_layer
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136,claire.gute@gmail.com,8235062526,2000-03-02,5527812096789655,08/28,2026-07-07T02:04:47.542478,retail_pos_ecommerce,bronze
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820,claire.gute@gmail.com,8235062526,2000-03-02,5527812096789655,08/28,2026-07-07T02:04:47.542478,retail_pos_ecommerce,bronze
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714,darrin.van.huff@gmail.com,4754940413,1988-08-02,5465283994305638,11/26,2026-07-07T02:04:47.542478,retail_pos_ecommerce,bronze


Bronze **still contains PII/PCI** for lineage/audit purposes, but CVV is permanently gone. Access to Bronze is restricted to the `compliance_admin` role only (enforced in Step 5).

## Step 3: Silver Layer — Masking, Tokenization, Encryption & Feature Engineering

This is where raw identifiers are transformed into safe, analytics-friendly forms.


In [25]:
HASH_SALT = "celebal-retail-lakehouse-2026"  # in production: Azure Key Vault secret

def mask_name(name):
    return " ".join(p[0] + "*" * (len(p) - 1) for p in str(name).split())

def mask_email(email):
    local, _, domain = str(email).partition("@")
    masked_local = local[0] + "*" if len(local) <= 2 else local[:2] + "*" * (len(local) - 2)
    return f"{masked_local}@{domain}"

def mask_phone(phone):
    digits = "".join(ch for ch in str(phone) if ch.isdigit())
    return "*" * len(digits) if len(digits) < 4 else "X" * (len(digits) - 4) + digits[-4:]

def tokenize_card(card_number):
    salted = f"{HASH_SALT}:{card_number}"
    return "TOK-" + hashlib.sha256(salted.encode()).hexdigest()[:20]

def mask_card_last4(card_number):
    card_number = str(card_number)
    return "X" * (len(card_number) - 4) + card_number[-4:]

def age_bucket(age):
    if age < 25: return "18-24"
    elif age < 35: return "25-34"
    elif age < 45: return "35-44"
    elif age < 60: return "45-59"
    return "60+"

def amount_bucket(amount):
    if amount < 50: return "Low (<50)"
    elif amount < 500: return "Medium (50-500)"
    elif amount < 1500: return "High (500-1500)"
    return "Very High (1500+)"

print("Masking / tokenization / bucketing functions defined.")


Masking / tokenization / bucketing functions defined.


In [26]:
# Demonstrate masking on a sample before running the full pipeline
sample_name = bronze_df['full_name'].iloc[0]
sample_email = bronze_df['email'].iloc[0]
sample_card = bronze_df['card_number'].iloc[0]

print(f"Name:  {sample_name}  ->  {mask_name(sample_name)}")
print(f"Email: {sample_email}  ->  {mask_email(sample_email)}")
print(f"Card:  {sample_card}  ->  masked: {mask_card_last4(sample_card)}  |  token: {tokenize_card(sample_card)}")


Name:  Claire Gute  ->  C***** G***
Email: claire.gute@gmail.com  ->  cl*********@gmail.com
Card:  5527812096789655  ->  masked: XXXXXXXXXXXX9655  |  token: TOK-a83cc3f6ccc5644fd900


In [27]:
def get_or_create_key(key_path):
    import os
    if os.path.exists(key_path):
        with open(key_path, "rb") as f:
            return f.read()
    key = Fernet.generate_key()
    os.makedirs(os.path.dirname(key_path), exist_ok=True)
    with open(key_path, "wb") as f:
        f.write(key)
    print(f"[ENCRYPTION] New key generated at {key_path} (simulates Azure Key Vault).")
    return key

def transform_to_silver(bronze_path, silver_path, key_path):
    df = pd.read_parquet(bronze_path)
    key = get_or_create_key(key_path)
    fernet = Fernet(key)

    # MASKING
    df["full_name_masked"] = df["full_name"].apply(mask_name)
    df["email_masked"] = df["email"].apply(mask_email)
    df["phone_masked"] = df["phone_number"].apply(mask_phone)
    df["card_number_masked"] = df["card_number"].apply(mask_card_last4)

    # TOKENIZATION
    df["card_token"] = df["card_number"].apply(tokenize_card)

    # ENCRYPTION (defense-in-depth on top of tokenization)
    df["card_token_encrypted"] = df["card_token"].apply(lambda t: fernet.encrypt(t.encode()).decode())

    # FEATURE ENGINEERING
    df["date_of_birth"] = pd.to_datetime(df["date_of_birth"])
    today = pd.Timestamp(datetime.utcnow().date())
    df["age"] = ((today - df["date_of_birth"]).dt.days // 365).astype(int)
    df["age_bucket"] = df["age"].apply(age_bucket)
    df["amount_bucket"] = df["sales"].apply(amount_bucket)

    # DROP raw sensitive columns — Silver never stores raw PII/PCI
    drop_cols = ["full_name", "email", "phone_number",
                 "date_of_birth", "card_number", "card_expiry", "card_token", "postal_code"]
    df_silver = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df_silver["_layer"] = "silver"

    df_silver.to_parquet(silver_path, index=False)
    print(f"[SILVER] {df_silver.shape[0]} rows, {df_silver.shape[1]} columns -> {silver_path}")
    return df_silver

silver_df = transform_to_silver("bronze/retail_bronze.parquet", "silver/retail_silver.parquet", "keys/encryption.key")
silver_df.head(3)


[SILVER] 9994 rows, 30 columns -> silver/retail_silver.parquet


C:\Users\Admin\AppData\Local\Temp\ipykernel_24084\4288565077.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = pd.Timestamp(datetime.utcnow().date())


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,segment,country,city,state,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,_ingestion_timestamp,_source_system,_layer,full_name_masked,email_masked,phone_masked,card_number_masked,card_token_encrypted,age,age_bucket,amount_bucket
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Consumer,United States,Henderson,Kentucky,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136,2026-07-07T02:04:47.542478,retail_pos_ecommerce,silver,C***** G***,cl*********@gmail.com,XXXXXX2526,XXXXXXXXXXXX9655,gAAAAABqTF7AVG4bnEvR4B1rjdO5dt4veS0iEJRO9RqZ4R...,26,25-34,Medium (50-500)
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Consumer,United States,Henderson,Kentucky,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820,2026-07-07T02:04:47.542478,retail_pos_ecommerce,silver,C***** G***,cl*********@gmail.com,XXXXXX2526,XXXXXXXXXXXX9655,gAAAAABqTF7Ae8D6R20cvUr745vtqfMatetIYNyWoAoeCz...,26,25-34,High (500-1500)
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Corporate,United States,Los Angeles,California,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714,2026-07-07T02:04:47.542478,retail_pos_ecommerce,silver,D***** V** H***,da*************@gmail.com,XXXXXX0413,XXXXXXXXXXXX5638,gAAAAABqTF7AX1YKgWhJPUtHZkKRKmuZ9a2zq2apsYMOzH...,37,35-44,Low (<50)


**Notice what's gone from Silver:** `full_name`, `email`, `phone_number`, `date_of_birth`,
`card_number`, `card_expiry`, `postal_code`, and the plaintext `card_token` are **all dropped**.
Only masked/tokenized/encrypted/derived versions remain.


## Step 4: Gold Layer — Fully Anonymized Aggregates

Business analysts and most data scientists should never see row-level customer data at all.
Gold aggregates everything by safe dimensions (month, state, category, age bucket, amount
bucket) — no way to trace a row back to an individual.


In [28]:
def build_gold(silver_path, gold_path):
    df = pd.read_parquet(silver_path)
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["transaction_month"] = df["order_date"].dt.to_period("M").astype(str)

    gold = (
        df.groupby(["transaction_month", "state", "category", "age_bucket", "amount_bucket"])
        .agg(
            total_transactions=("order_id", "count"),
            total_revenue=("sales", "sum"),
            total_profit=("profit", "sum"),
            avg_transaction_value=("sales", "mean"),
            unique_customers=("customer_id", "nunique"),
        )
        .reset_index()
    )
    gold["total_revenue"] = gold["total_revenue"].round(2)
    gold["total_profit"] = gold["total_profit"].round(2)
    gold["avg_transaction_value"] = gold["avg_transaction_value"].round(2)

    gold.to_csv(gold_path, index=False)
    print(f"[GOLD] {gold.shape[0]} rows, {gold.shape[1]} columns -> {gold_path}")
    return gold

gold_df = build_gold("silver/retail_silver.parquet", "gold/retail_gold_aggregated.csv")
gold_df.head(5)


[GOLD] 6482 rows, 10 columns -> gold/retail_gold_aggregated.csv


,transaction_month,state,category,age_bucket,amount_bucket,total_transactions,total_revenue,total_profit,avg_transaction_value,unique_customers
0,2014-01,Arizona,Furniture,60+,Medium (50-500),1,181.47,-320.60,181.47,1
1,2014-01,Arizona,Office Supplies,60+,Low (<50),1,32.34,-23.72,32.34,1
2,2014-01,Arizona,Office Supplies,60+,Medium (50-500),2,164.78,56.32,82.39,1
3,2014-01,Arkansas,Furniture,45-59,High (500-1500),1,1067.94,224.27,1067.94,1
4,2014-01,Arkansas,Furniture,45-59,Low (<50),1,38.60,11.58,38.60,1


In [29]:
# Quick analytics example an analyst COULD run on Gold — no identifiers needed
top_categories = gold_df.groupby("category")[["total_revenue", "total_profit"]].sum().sort_values("total_revenue", ascending=False)
print("Revenue & profit by category (fully safe, aggregated view):")
top_categories


Revenue & profit by category (fully safe, aggregated view):


,total_revenue,total_profit
category,,
Technology,836154.01,145455.22
Furniture,741999.60,18451.17
Office Supplies,719046.78,122490.63


## Step 5: Access Control (RBAC)

Simulates what Azure Data Lake ACLs / Databricks Unity Catalog GRANTs would enforce in
production — different roles see different layers/columns.

| Role | Layer Access | Can Decrypt Card Token? |
|------|--------------|--------------------------|
| `business_analyst` | Gold only | No |
| `data_scientist` | Silver, excluding encrypted token | No |
| `data_engineer` | Silver, full | No |
| `compliance_admin` | Bronze + decryption rights | Yes (fraud investigation only) |


In [30]:
ROLE_POLICY = {
    "business_analyst": {"layer": "gold", "columns": "all", "can_decrypt": False},
    "data_scientist":   {"layer": "silver", "columns": "exclude_encrypted", "can_decrypt": False},
    "data_engineer":    {"layer": "silver", "columns": "all", "can_decrypt": False},
    "compliance_admin": {"layer": "bronze", "columns": "all", "can_decrypt": True},
}

def get_data_for_role(role):
    if role not in ROLE_POLICY:
        raise PermissionError(f"Unknown role '{role}'. Access denied.")
    policy = ROLE_POLICY[role]
    layer = policy["layer"]

    if layer == "gold":
        df = pd.read_csv("gold/retail_gold_aggregated.csv")
    elif layer == "silver":
        df = pd.read_parquet("silver/retail_silver.parquet")
        if policy["columns"] == "exclude_encrypted":
            df = df.drop(columns=["card_token_encrypted"])
    elif layer == "bronze":
        df = pd.read_parquet("bronze/retail_bronze.parquet")

    print(f"[ACCESS GRANTED] role='{role}' -> layer='{layer}' -> {df.shape[0]} rows, {df.shape[1]} cols")
    return df

def decrypt_card_token(role, encrypted_token):
    if not ROLE_POLICY.get(role, {}).get("can_decrypt", False):
        raise PermissionError(f"Role '{role}' is not authorized to decrypt payment tokens.")
    with open("keys/encryption.key", "rb") as f:
        key = f.read()
    return Fernet(key).decrypt(encrypted_token.encode()).decode()


In [31]:
# Demo: business_analyst
analyst_view = get_data_for_role("business_analyst")
analyst_view.head(2)


[ACCESS GRANTED] role='business_analyst' -> layer='gold' -> 6482 rows, 10 cols


,transaction_month,state,category,age_bucket,amount_bucket,total_transactions,total_revenue,total_profit,avg_transaction_value,unique_customers
0,2014-01,Arizona,Furniture,60+,Medium (50-500),1,181.47,-320.60,181.47,1
1,2014-01,Arizona,Office Supplies,60+,Low (<50),1,32.34,-23.72,32.34,1


In [32]:
# Demo: data_scientist (no encrypted token visible)
ds_view = get_data_for_role("data_scientist")
print("Columns visible to data_scientist:", list(ds_view.columns))


[ACCESS GRANTED] role='data_scientist' -> layer='silver' -> 9994 rows, 29 cols
Columns visible to data_scientist: ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'segment', 'country', 'city', 'state', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', '_ingestion_timestamp', '_source_system', '_layer', 'full_name_masked', 'email_masked', 'phone_masked', 'card_number_masked', 'age', 'age_bucket', 'amount_bucket']


In [33]:
# Demo: data_scientist tries to decrypt a card token -> DENIED
sample_token = pd.read_parquet("silver/retail_silver.parquet")["card_token_encrypted"].iloc[0]
try:
    decrypt_card_token("data_scientist", sample_token)
except PermissionError as e:
    print(f"[ACCESS DENIED] {e}")


[ACCESS DENIED] Role 'data_scientist' is not authorized to decrypt payment tokens.


In [34]:
# Demo: compliance_admin decrypts the same token -> SUCCEEDS (fraud investigation use case)
decrypted = decrypt_card_token("compliance_admin", sample_token)
print(f"[ACCESS GRANTED] Decrypted surrogate token: {decrypted}")


[ACCESS GRANTED] Decrypted surrogate token: TOK-a83cc3f6ccc5644fd900


## Step 6: Compliance Mapping

| Regulation | Requirement | How This Pipeline Satisfies It |
|------------|-------------|----------------------------------|
| **PCI-DSS** | Never store CVV under any circumstance | Hard-dropped at Bronze ingestion |
| **PCI-DSS** | Render PAN (card number) unreadable when stored | Tokenized (SHA-256) + encrypted (Fernet/AES-128) |
| **GDPR** | Data minimization — only collect/retain what's needed | Gold retains zero identifiers, only aggregates |
| **GDPR** | Right to be forgotten (support deletion) | Customer ID is the only linking key in Silver; deleting Bronze/Silver rows by ID satisfies erasure |
| **GDPR / DPDP** | Purpose limitation — analysts get only what they need | RBAC restricts business analysts to Gold-only aggregated view |
| **DPDP (India)** | Reasonable security safeguards for personal data | Encryption at rest + masking + access control combined (defense-in-depth) |


## Summary

| Layer | Rows | Columns | Contains PII? | Contains PCI? | Who Accesses |
|-------|------|---------|----------------|-----------------|----------------|
| Raw | 9,994 | 27 | Yes (plaintext) | Yes (plaintext, incl. CVV) | No one (transient) |
| Bronze | 9,994 | 29 | Yes (plaintext, CVV dropped) | Partial (CVV gone) | `compliance_admin` |
| Silver | 9,994 | 30 | No (masked only) | No (tokenized + encrypted) | `data_engineer`, `data_scientist` |
| Gold | 6,473 | 10 | No | No | `business_analyst` (everyone) |

**Security controls demonstrated:** hard-drop, masking, salted one-way tokenization, symmetric
encryption (Fernet/AES-128) with key separation, feature engineering for privacy-safe
analytics (age/amount bucketing), and role-based access control (RBAC) — applied on top of
the **real Superstore transactional dataset**.
